
Project : Pentaho Log Intelligence

Layer   : GOLD

Notebook: 01_Gold_Log_Carte

Version : 1.0

Description:
Loads raw Pentaho log files from Unity Catalog Volume

into the GOLD Delta table.

Author: Ernesto Felipe Garay Cervantes


#### Recibimiento de Parametros

In [0]:
import json

dbutils.widgets.text("archivos_nuevos","")

archivos_nuevos = json.loads(dbutils.widgets.get("archivos_nuevos"))

print("====ARCHIVOS RECIBIDOS===")
for archivo in archivos_nuevos:
    print(archivo)

 
#dbutils.notebook.exit("notebook hijo parametro que recibe : " + archivo)

#### Configuracion

In [0]:
from pyspark.sql.functions import (
    col,
    lit,
    sha2,
    concat_ws,
    current_timestamp
)

In [0]:
CATALOG = "pentaho_logs"

SILVER_TABLE_CARTE = "pentaho_logs.silver.silver_logs_carte"

GOLD_TABLE_CARTE = "pentaho_logs.gold.gold_logs_carte"

#### Lectura  de tabla SILVER 

In [0]:
df_carte_silver = spark.table(SILVER_TABLE_CARTE).filter(col("file_name").isin(archivos_nuevos))

#display(df_carte_silver.limit(20))

In [0]:
df_carte_silver.printSchema()

#### Creación de identificador de Evento CARTE

In [0]:
from pyspark.sql.functions import (col,lit,sha2,concat_ws,current_timestamp)


df_gold_carte = (df_carte_silver .withColumn("event_id",sha2(concat_ws("||",col("file_path"),col("tiempo_evento"),col("Nivel_log"),col("Thread"),col("Descripcion")),256))
                 
                 .withColumn("source_type",lit("CARTE"))
                  .withColumn("gold_timestamp",current_timestamp())
                 
                 )



In [0]:
#display(
    #df_gold_carte.select(
      #  "event_id",
      #  "file_name",
       # "application",
      #  "server_port",
     #   "log_date",
    #    "tiempo_evento",
   #     "Nivel_log",
  #      "Descripcion"
 #   ).limit(20)
#)

In [0]:
df_gold_carte = df_gold_carte.select(
    "event_id",
    "file_name",
    "application",
    "server_port",
    "log_date",
    "tiempo_evento",
    "Nivel_log",
    "Descripcion"
)

In [0]:
#display(df_gold_carte.limit(20))

In [0]:
df_gold_carte.printSchema()


#### validación DATAFRAME

In [0]:
print(f"Registros Gold: {df_gold_carte.count():,}")

In [0]:
from pyspark.sql.functions import col, sum, when

df_gold_carte.select(
    sum(when(col("event_id").isNull(), 1).otherwise(0)).alias("event_id_null"),
    sum(when(col("file_name").isNull(), 1).otherwise(0)).alias("file_name_null"),
    sum(when(col("application").isNull(), 1).otherwise(0)).alias("application_null"),
    sum(when(col("log_date").isNull(), 1).otherwise(0)).alias("log_date_null"),
    sum(when(col("tiempo_evento").isNull(), 1).otherwise(0)).alias("tiempo_evento_null"),
    sum(when(col("Nivel_log").isNull(), 1).otherwise(0)).alias("Nivel_log_null"),
    sum(when(col("Descripcion").isNull(), 1).otherwise(0)).alias("Descripcion_null")
).show()


#### Creación Tabla Gold Carte 

In [0]:
GOLD_TABLE_CARTE = "pentaho_logs.gold.gold_logs_carte"
(
    df_gold_carte.write
        .format("delta")
        .mode("append")
        .saveAsTable(GOLD_TABLE_CARTE)
)

In [0]:
#display(spark.table(GOLD_TABLE_CARTE).limit(20))